# Transaction Classification

This notebook loads local bank exports.

## 0. Imports & Setup

In [ ]:
# Step 0: Imports
import json
import sqlite3
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

OLLAMA_MODEL = "gemma3:4b"
OLLAMA_URL = "http://localhost:11434/api/generate"
VENDOR_EXTRACTION_BATCH_SIZE = 10
TRANSACTION_CLASSIFICATION_BATCH_SIZE = 8


## 1. Load transactions

Every semicolon-delimited CSV in `../data` is appended to one dataframe.

In [ ]:
def load_transactions_csv(csv_path):
    """Load a semicolon-delimited bank export as text."""
    try:
        return pd.read_csv(
            csv_path,
            sep=";",
            encoding="utf-8-sig",
            dtype="string",
            keep_default_na=False,
        )
    except (FileNotFoundError, UnicodeDecodeError, pd.errors.ParserError) as error:
        raise ValueError(f"Could not load CSV: {error}") from error


data_folder = Path("../data")
transactions = pd.DataFrame()

for csv_path in sorted(data_folder.glob("*.csv")):
    file_transactions = load_transactions_csv(csv_path)
    transactions = pd.concat([transactions, file_transactions], ignore_index=True)

transactions.head()


## 2. Extract vendors

All transactions are sent to local Ollama in batches of 10. (this step may take up to 1 hour)

In [ ]:
def extract_vendors_batch(batch):
    batch_description = "\n\n".join(
        f"Transaction {number}:\n"
        + "\n".join(
            f"{column}: {transaction[column]}"
            for column in ["Beschreibung1", "Beschreibung2", "Beschreibung3"]
            if transaction[column]
        )
        for number, (_, transaction) in enumerate(batch.iterrows(), start=1)
    )

    prompt = (
        "Extract one vendor for each numbered bank transaction. Keep the vendor spelling "
        "from the description. For interest credits or service fees, the bank can be the vendor. "
        "Do not assign categories. Return JSON only, for example: "
        '{\"vendors\": {\"1\": \"Vendor A\", \"2\": \"Vendor B\"}}.\n\n'
        + batch_description
    )

    try:
        response = requests.post(
            OLLAMA_URL,
            json={"model": OLLAMA_MODEL, "prompt": prompt, "format": "json", "stream": False},
            timeout=60,
        )
        response.raise_for_status()
        vendors = json.loads(response.json()["response"])["vendors"]
        return [vendors[str(number)].strip() for number in range(1, len(batch) + 1)]
    except (requests.RequestException, json.JSONDecodeError, KeyError, AttributeError) as error:
        raise ValueError(f"Could not extract vendors: {error}") from error


database_path = Path("../database/vendor_mappings.sqlite")
transaction_batches = []

for start_index in range(0, len(transactions), VENDOR_EXTRACTION_BATCH_SIZE):
    batch = transactions.iloc[start_index : start_index + VENDOR_EXTRACTION_BATCH_SIZE].copy()
    batch["vendor"] = extract_vendors_batch(batch)
    transaction_batches.append(batch)

    database_batch = batch
    with sqlite3.connect(database_path) as connection:
        database_batch.to_sql(
            "transactions",
            connection,
            if_exists="replace" if start_index == 0 else "append",
            index=False,
        )

transactions = pd.concat(transaction_batches, ignore_index=True)

display(transactions[["Beschreibung1", "Beschreibung2", "Beschreibung3", "vendor"]])


## 3. Load transactions from the database

In [ ]:
database_path = Path("../database/vendor_mappings.sqlite")

with sqlite3.connect(database_path) as connection:
    transactions = pd.read_sql_query("SELECT * FROM transactions", connection)

transactions.head()


## 4. Categorize vendors

Known vendor mappings are reused from the database. Only new vendors are classified before categories are assigned to transactions.

In [ ]:
category_path = Path("../categories.txt")

categories = [
    line.strip()
    for line in category_path.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.strip().startswith("#")
]

distinct_vendors = transactions[["vendor"]].drop_duplicates().reset_index(drop=True)

if distinct_vendors.empty:
    raise ValueError("No vendors were loaded from the database.")


def classify_vendors_batch(batch):
    vendor_list = "\n".join(
        f"{number}. {transaction['vendor']}"
        for number, (_, transaction) in enumerate(batch.iterrows(), start=1)
    )

    prompt = (
        "Classify each numbered transaction vendor into exactly one category. "
        "Use only the listed categories and choose Uncategorized when the vendor is unclear. "
        "Also return a confidence score from 0.00 to 1.00, with at most two decimal places, based on the vendor name alone. "
        "Always make sure to return results for every input transaction you got. "
        "Return JSON only, for example: {\"categories\": {\"1\": {\"category\": \"Groceries\", \"confidence\": 0.95}}}.\n\n"
        + "Categories:\n- " + "\n- ".join(categories)
        + "\n\nVendors:\n" + vendor_list
    )

    ollama_output = "No Ollama output was received."

    try:
        response = requests.post(
            OLLAMA_URL,
            json={"model": OLLAMA_MODEL, "prompt": prompt, "format": "json", "stream": False},
            timeout=60,
        )
        ollama_output = response.text
        response.raise_for_status()
        ollama_output = response.json().get("response", ollama_output)
        classified = json.loads(ollama_output)["categories"]
        result = [
            {
                "category": classified[str(number)]["category"].strip(),
                "confidence": round(float(classified[str(number)]["confidence"]), 2),
            }
            for number in range(1, len(batch) + 1)
        ]
        if not {item["category"] for item in result}.issubset(categories):
            raise ValueError("Ollama returned a category outside categories.txt.")
        if not all(0 <= item["confidence"] <= 1 for item in result):
            raise ValueError("Ollama returned a confidence outside the 0-1 range.")
        return result, prompt, ollama_output, None
    except (requests.RequestException, json.JSONDecodeError, KeyError, AttributeError, TypeError, ValueError) as error:
        return None, prompt, ollama_output, error
with sqlite3.connect(database_path) as connection:
    connection.execute(
        """
        CREATE TABLE IF NOT EXISTS vendor_category_mappings (
            vendor TEXT PRIMARY KEY,
            category TEXT NOT NULL,
            category_confidence REAL NOT NULL
        )
        """
    )
    connection.execute(
        "DELETE FROM vendor_category_mappings "
        "WHERE rowid NOT IN ("
        "SELECT MAX(rowid) FROM vendor_category_mappings GROUP BY vendor"
        ")"
    )
    connection.execute(
        "CREATE UNIQUE INDEX IF NOT EXISTS idx_vendor_category_mappings_vendor "
        "ON vendor_category_mappings(vendor)"
    )
    known_vendors = {
        row[0] for row in connection.execute("SELECT vendor FROM vendor_category_mappings")
    }

vendors_to_classify = distinct_vendors.loc[
    ~distinct_vendors["vendor"].isin(known_vendors)
].copy()
print(
    f"Distinct vendors in transactions: {len(distinct_vendors)}\n"
    f"Already classified: {len(distinct_vendors) - len(vendors_to_classify)}\n"
    f"Still to classify: {len(vendors_to_classify)}"
)

total_batches = (
    len(vendors_to_classify) + TRANSACTION_CLASSIFICATION_BATCH_SIZE - 1
) // TRANSACTION_CLASSIFICATION_BATCH_SIZE

for batch_number, start_index in enumerate(
    range(0, len(vendors_to_classify), TRANSACTION_CLASSIFICATION_BATCH_SIZE), start=1
):
    batch = vendors_to_classify.iloc[
        start_index : start_index + TRANSACTION_CLASSIFICATION_BATCH_SIZE
    ].copy()
    classifications, model_input, model_output, error = classify_vendors_batch(batch)

    if error:
        print(
            f"Batch {batch_number}/{total_batches}: failed; no mappings saved.\n\n"
            f"Model input:\n{model_input}\n\n"
            f"Model output:\n{model_output}\n\n"
            f"Error:\n{error}"
        )
        continue

    batch["category"] = [item["category"] for item in classifications]
    batch["category_confidence"] = [item["confidence"] for item in classifications]

    try:
        with sqlite3.connect(database_path) as connection:
            connection.executemany(
                """
                INSERT INTO vendor_category_mappings
                    (vendor, category, category_confidence)
                VALUES (?, ?, ?)
                ON CONFLICT(vendor) DO UPDATE SET
                    category = excluded.category,
                    category_confidence = excluded.category_confidence
                """,
                [
                    (row.vendor, row.category, row.category_confidence)
                    for row in batch.itertuples(index=False)
                ],
            )
        error = None
        status = "worked; mappings saved"
    except sqlite3.Error as save_error:
        error = save_error
        status = "failed; no mappings saved"

    print(f"Batch {batch_number}/{total_batches}: {status}.")
    if error:
        print(
            f"Model input:\n{model_input}\n\n"
            f"Model output:\n{model_output}\n\n"
            f"Error:\n{error}"
        )

with sqlite3.connect(database_path) as connection:
    vendor_category_mappings = pd.read_sql_query(
        "SELECT * FROM vendor_category_mappings ORDER BY vendor", connection
    )

display(vendor_category_mappings)


## 5. Review vendor classifications

Review vendors that are Uncategorized or have a confidence score below 0.85. Choose a category and save it to move to the next vendor. Manually reviewed vendors leave the queue; the original model confidence is preserved.

In [ ]:
import sqlite3
from html import escape
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

database_path = Path("../database/vendor_mappings.sqlite").resolve()
if not database_path.exists():
    raise ValueError(f"Database not found: {database_path.resolve()}")

review_categories = [
    line.strip()
    for line in Path("../categories.txt").read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.strip().startswith("#")
]

with sqlite3.connect(database_path) as connection:
    columns = {
        row[1] for row in connection.execute("PRAGMA table_info(vendor_category_mappings)")
    }
    if "manually_reviewed" not in columns:
        connection.execute(
            "ALTER TABLE vendor_category_mappings "
            "ADD COLUMN manually_reviewed INTEGER NOT NULL DEFAULT 0"
        )
    vendors_to_review = pd.read_sql_query(
        """
        SELECT vendor, category, category_confidence
        FROM vendor_category_mappings
        WHERE COALESCE(manually_reviewed, 0) = 0
          AND (category = ? OR category_confidence < ?)
        ORDER BY category_confidence, vendor
        """,
        connection,
        params=("Uncategorized", 0.85),
    )

review_state = {"index": 0}
progress = widgets.Label()
review_output = widgets.HTML()
category_dropdown = widgets.Dropdown(
    options=review_categories, description="Category:", layout=widgets.Layout(width="500px")
)
save_button = widgets.Button(description="Save and next", button_style="success")
review_status = widgets.Label()


def show_vendor_to_review():
    review_output.value = ""
    if review_state["index"] >= len(vendors_to_review):
        progress.value = "Review complete. No vendors remain in the queue."
        category_dropdown.disabled = True
        save_button.disabled = True
        return

    vendor = vendors_to_review.iloc[review_state["index"]]
    progress.value = f"Vendor {review_state['index'] + 1} of {len(vendors_to_review)}"
    category_dropdown.value = (
        vendor.category if vendor.category in review_categories else "Uncategorized"
    )
    save_button.disabled = False
    with sqlite3.connect(database_path) as connection:
        examples = pd.read_sql_query(
            """
            SELECT Buchungsdatum, Beschreibung1, Beschreibung2, Beschreibung3
            FROM transactions WHERE vendor = ? LIMIT 5
            """,
            connection,
            params=(vendor.vendor,),
        )
    review_output.value = (
        f"<h3>Vendor: {escape(str(vendor.vendor))}</h3>"
        f"<p>Current category: {escape(str(vendor.category))}; "
        f"model confidence: {vendor.category_confidence}</p>"
        + examples.to_html(index=False, escape=True)
    )


def save_vendor_review(_):
    save_button.disabled = True
    vendor = vendors_to_review.iloc[review_state["index"]].vendor
    selected_category = category_dropdown.value
    review_status.value = f"Saving {vendor}..."
    try:
        with sqlite3.connect(database_path) as connection:
            cursor = connection.execute(
                "UPDATE vendor_category_mappings "
                "SET category = ?, manually_reviewed = 1 WHERE vendor = ?",
                (selected_category, vendor),
            )
            if cursor.rowcount != 1:
                raise sqlite3.IntegrityError(
                    f"Expected to update one mapping for {vendor}; found {cursor.rowcount}."
                )
            connection.commit()

        with sqlite3.connect(database_path) as verification_connection:
            saved_mapping = verification_connection.execute(
                "SELECT category, manually_reviewed FROM vendor_category_mappings WHERE vendor = ?",
                (vendor,),
            ).fetchone()
        if saved_mapping != (selected_category, 1):
            raise sqlite3.DatabaseError(
                "Save verification failed: the stored mapping does not match the selection."
            )
    except sqlite3.Error as error:
        review_status.value = f"Could not save {vendor}: {error}"
        save_button.disabled = False
        return

    review_status.value = f"Saved and verified {vendor} as {selected_category}."
    review_state["index"] += 1
    show_vendor_to_review()


save_button.on_click(save_vendor_review)
show_vendor_to_review()
review_interface = widgets.VBox(
    [progress, review_output, category_dropdown, save_button, review_status]
)
print(f"Vendors to review: {len(vendors_to_review)}")
print(f"Saving reviews to: {database_path}")
print("If the controls below are missing, restart the kernel, refresh Jupyter, and run Step 5 again.")
display(review_interface)


## 6. Apply vendor categories to transactions

Replace the `categorized_transactions` table with all transactions plus their vendor category and confidence.

In [ ]:
import sqlite3
from pathlib import Path

database_path = Path("../database/vendor_mappings.sqlite").resolve()
if not database_path.exists():
    raise ValueError(f"Database not found: {database_path}")

with sqlite3.connect(database_path) as connection:
    connection.execute("BEGIN")
    connection.execute("DROP TABLE IF EXISTS categorized_transactions")
    connection.execute(
        """
        CREATE TABLE categorized_transactions AS
        SELECT t.*, m.category, m.category_confidence
        FROM transactions AS t
        LEFT JOIN vendor_category_mappings AS m ON t.vendor = m.vendor
        """
    )

print("categorized_transactions saved.")


## 7. Export categorized transactions

Export the table to `../output/categorized_transactions.csv` using semicolons and UTF-8 encoding with a BOM. Each run replaces the CSV.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

database_path = Path("../database/vendor_mappings.sqlite").resolve()
if not database_path.exists():
    raise ValueError(f"Database not found: {database_path}")

output_path = Path("../output/categorized_transactions.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

with sqlite3.connect(database_path) as connection:
    export_transactions = pd.read_sql_query(
        "SELECT * FROM categorized_transactions", connection
    )

export_transactions.to_csv(output_path, sep=";", index=False, encoding="utf-8-sig")
print(f"CSV exported to: {output_path.resolve()}")
